In [1]:
# Chapter 6 Case Study 2: Stock Price Fluctuation Prediction — Random Forest Classifier

import numpy as np
import yfinance as yf           # yfinance is a library to download financial data from Yahoo Finance
import talib                    # Technical Analysis Library, used to compute technical indicators in financial markets
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Data Acquisition & Preprocessing ─────────────────────────────────────────
# Download historical daily OHLCV data for SANY Heavy Industry (600031.SS)
df = yf.download('600031.SS', start='2012-03-01', end='2022-03-01', auto_adjust=True)

# Flatten MultiIndex columns returned by recent yfinance versions, then lowercase
df.columns = df.columns.get_level_values(0).str.lower()

print(df.head())

# Feature engineering: intraday price range ratios
df['close_open'] = (df['close'] - df['open']) / df['open']   # Close-to-open return
df['high_low']   = (df['high']  - df['low'])  / df['low']    # High-to-low range ratio

# Daily price change (close minus previous close); shift(1) moves the series down by 1 row
df['price_change'] = df['close'] - df['close'].shift(1)

# Rolling moving averages
df['m5']  = df['close'].rolling(5).mean()    # 5-day moving average
df['m10'] = df['close'].rolling(10).mean()   # 10-day moving average

# RSI (Relative Strength Index) with a 12-period window
df['rsi'] = talib.RSI(df['close'], timeperiod=12)

# Drop rows with NaN values introduced by rolling windows and RSI warm-up period
df.dropna(inplace=True)

# ── Feature & Target Variable Extraction ──────────────────────────────────────
# Features: technical indicators used as model inputs
X = df[['close', 'volume', 'high_low', 'close_open', 'm5', 'm10', 'rsi']]

# Target: 1 if next-day price rises (price_change > 0), else -1
# shift(-1) looks one day ahead to label the current row
Y = np.where(df['price_change'].shift(-1) > 0, 1, -1)

X_len = X.shape[0]
print('Total number of samples: {}'.format(X_len))

# ── Train / Test Split (80 / 20, time-ordered) ────────────────────────────────
split = int(X_len * 0.8)
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

# ── Model Training ────────────────────────────────────────────────────────────
# Random Forest with manually tuned hyperparameters
model = RandomForestClassifier(
    max_depth=6,            # Limit tree depth to reduce overfitting
    n_estimators=8,         # Number of trees in the forest
    min_samples_leaf=10,    # Minimum samples required at a leaf node
    random_state=2          # Fixed seed for reproducibility
)
model.fit(X_train, Y_train)

# ── Prediction & Evaluation ───────────────────────────────────────────────────
Y_predict = model.predict(X_test)

# Show all rows when printing DataFrames
pd.set_option('display.max_rows', None)
print(X_test)
print(Y_test)
print(Y_predict)

# F1-score (binary): harmonic mean of precision and recall
# pos_label defaults to 1, so this measures performance on the 'price up' class
num = f1_score(Y_test, Y_predict, average='binary', pos_label=1)
print('F1-score: ' + str(num))

# ── Feature Importance Ranking ────────────────────────────────────────────────
features    = X.columns
importances = model.feature_importances_

# Build a DataFrame for readable display, sorted by importance descending
importances_df = pd.DataFrame({
    'Feature':    features,
    'Importance': importances
})
print(importances_df.sort_values('Importance', ascending=False))

[*********************100%***********************]  1 of 1 completed

Price           close       high        low       open    volume
Date                                                            
2012-03-01  11.311136  11.381150  11.280019  11.342254  15431636
2012-03-02  11.544516  11.552295  11.303356  11.311136  35637514
2012-03-05  11.560074  11.785674  11.536736  11.575633  40691966
2012-03-06  11.217784  11.521178  11.202225  11.521178  27886287
2012-03-07  10.961066  11.163329  10.929949  11.108873  36351918
Total number of samples: 2416
Price           close     volume  high_low  close_open         m5        m10  \
Date                                                                           
2020-03-03  16.314219  188270365  0.075900   -0.042487  15.793365  15.406697   
2020-03-04  16.614372  110369116  0.040176    0.027854  16.132361  15.590320   
2020-03-05  16.640858  101602195  0.037675   -0.001060  16.335406  15.753639   
2020-03-06  16.111174   84760849  0.027624   -0.016172  16.480186  15.844568   
2020-03-09  15.484385   84266872  0